# Juicio de expertos, segunda ronda: coeficiente V de Aiken

Validez de contenido de las variables predictoras, las preguntas del formulario, los mensajes del resultado y el catálogo de recomendaciones (plan de tesis, 4.7).

- **Ítems:** `juicio_expertos/ronda2/items.csv` (los 12 ítems corregidos tras la primera ronda), generados a partir de los textos del sistema.
- **Calificaciones:** `juicio_expertos/ronda2/calificaciones.xlsx`, con una hoja por experto (E1 a E4) y la misma disposición que la ficha: valores de 1 a 4 por ítem y criterio, observaciones por ítem, observaciones generales y opinión de aplicabilidad. Las hojas sin calificaciones se ignoran.
- **Coeficiente:** V = Σ(xᵢ − 1) / [n (c − 1)], con n expertos y c = 4 categorías (Aiken, 1985). Intervalo de confianza del 95 % por el método de puntuación de Penfield y Giacobbi (2004).
- **Criterio de validez (plan de tesis):** V ≥ 0.80. Los ítems con V < 0.80 se corrigen; cada cambio se registra con fecha en el protocolo y la versión del catálogo se actualiza.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import load_workbook

ITEMS = Path('../../juicio_expertos/ronda2/items.csv')
CALIFICACIONES = Path('../../juicio_expertos/ronda2/calificaciones.xlsx')
SALIDA = Path('../../resultados/juicio_expertos/ronda2')
C, Z, V_MINIMO = 4, 1.959964, 0.80

items = pd.read_csv(ITEMS)
ids = set(items['item_id'])


def leer_fichas(ruta):
    """Lee la plantilla de Excel (una hoja por experto) y devuelve las calificaciones en formato ancho
    (una fila por ítem y criterio, una columna por experto), las observaciones por ítem y las generales."""
    libro = load_workbook(ruta, data_only=True)
    filas, observaciones, generales = [], [], []
    for experto in [h for h in libro.sheetnames if h.startswith('E')]:
        criterios, general = None, {'experto': experto}
        for r in libro[experto].iter_rows(values_only=True):
            a = r[0]
            if a == 'N.°':
                criterios = [x for x in r[2:6] if x and x != '—']
            elif isinstance(a, str) and a in ids:
                filas += [(a, crit, experto, r[2 + k]) for k, crit in enumerate(criterios)]
                if r[6] not in (None, ''):
                    observaciones.append({'experto': experto, 'item_id': a, 'observacion': str(r[6]).strip()})
            elif a == 'E. Observaciones generales':
                general['observaciones_generales'] = r[2]
            elif a == 'Opinión de aplicabilidad':
                general['aplicabilidad'] = r[2]
        generales.append(general)
    largo = pd.DataFrame(filas, columns=['item_id', 'criterio', 'experto', 'puntaje'])
    largo['puntaje'] = pd.to_numeric(largo['puntaje'], errors='coerce')
    ancho = largo.pivot(index=['item_id', 'criterio'], columns='experto', values='puntaje').reset_index()
    orden = pd.DataFrame([(r.item_id, crit) for r in items.itertuples() for crit in r.criterios.split('|')],
                         columns=['item_id', 'criterio'])
    return orden.merge(ancho, on=['item_id', 'criterio'], how='left'), pd.DataFrame(observaciones), pd.DataFrame(generales)


cal, observaciones, generales = leer_fichas(CALIFICACIONES)
expertos = [c for c in cal.columns if c.startswith('E') and cal[c].notna().any()]
print(f'{len(items)} ítems, {len(cal)} filas de calificación | expertos con datos: {expertos or "ninguno"}')

12 ítems, 37 filas de calificación | expertos con datos: ['E1', 'E2', 'E3', 'E4']


## Comprobaciones de los datos

In [2]:
HAY_DATOS = bool(expertos)
if not HAY_DATOS:
    print('La plantilla todavía no tiene calificaciones. Complete juicio_expertos/ronda2/calificaciones.xlsx y vuelva a ejecutar el notebook.')
else:
    esperado = {(r.item_id, c) for r in items.itertuples() for c in r.criterios.split('|')}
    assert set(zip(cal.item_id, cal.criterio)) == esperado, 'las filas no coinciden con los ítems y criterios de items.csv'
    incompletos = {e: int(cal[e].isna().sum()) for e in expertos if cal[e].isna().any()}
    assert not incompletos, f'calificaciones vacías o no numéricas por experto: {incompletos}'
    valores = cal[expertos].to_numpy()
    assert np.isin(valores, [1, 2, 3, 4]).all(), 'hay calificaciones fuera de la escala de 1 a 4'
    SALIDA.mkdir(parents=True, exist_ok=True)
    cal[['item_id', 'criterio', *expertos]].to_csv(SALIDA / 'calificaciones.csv', index=False)
    print(f'Datos completos: {len(expertos)} expertos x {len(cal)} filas, todas en la escala 1-4.')

Datos completos: 4 expertos x 37 filas, todas en la escala 1-4.


## V de Aiken por ítem y criterio

Límites del intervalo de Penfield y Giacobbi (2004), con k = c − 1:

L, U = [2nkV + z² ∓ z √(4nkV(1 − V) + z²)] / [2(nk + z²)]

In [3]:
def v_aiken(x):
    x = np.asarray(x, dtype=float)
    n, k = len(x), C - 1
    v = (x - 1).sum() / (n * k)
    raiz = Z * np.sqrt(4 * n * k * v * (1 - v) + Z ** 2)
    return v, (2 * n * k * v + Z ** 2 - raiz) / (2 * (n * k + Z ** 2)), (2 * n * k * v + Z ** 2 + raiz) / (2 * (n * k + Z ** 2))


if HAY_DATOS:
    filas = []
    for r in cal.itertuples(index=False):
        x = [getattr(r, e) for e in expertos]
        v, li, ls = v_aiken(x)
        filas.append({'item_id': r.item_id, 'criterio': r.criterio, 'media': np.mean(x), 'V': v, 'IC95_inf': li, 'IC95_sup': ls,
                      'valido (V >= 0.80)': 'Sí' if v >= V_MINIMO else 'No'})
    resultado = pd.DataFrame(filas).merge(items[['item_id', 'seccion', 'nombre']], on='item_id')
    resultado = resultado[['seccion', 'item_id', 'nombre', 'criterio', 'media', 'V', 'IC95_inf', 'IC95_sup', 'valido (V >= 0.80)']]
    SALIDA.mkdir(parents=True, exist_ok=True)
    resultado.to_csv(SALIDA / 'v_aiken_por_criterio.csv', index=False)
    pd.set_option('display.float_format', '{:.3f}'.format)
    display(resultado)

,seccion,item_id,nombre,criterio,media,V,IC95_inf,IC95_sup,valido (V >= 0.80)
0,A. Variables predictoras,A1,Edad,Claridad,4.000,1.000,0.758,1.000,Sí
1,A. Variables predictoras,A1,Edad,Coherencia,4.000,1.000,0.758,1.000,Sí
2,A. Variables predictoras,A1,Edad,Relevancia,4.000,1.000,0.758,1.000,Sí
3,A. Variables predictoras,A2,Sexo,Claridad,4.000,1.000,0.758,1.000,Sí
4,A. Variables predictoras,A2,Sexo,Coherencia,3.750,0.917,0.646,0.985,Sí
5,A. Variables predictoras,A2,Sexo,Relevancia,4.000,1.000,0.758,1.000,Sí
6,A. Variables predictoras,A5,Hipertensión diagnosticada,Claridad,4.000,1.000,0.758,1.000,Sí
7,A. Variables predictoras,A5,Hipertensión diagnosticada,Coherencia,4.000,1.000,0.758,1.000,Sí
8,A. Variables predictoras,A5,Hipertensión diagnosticada,Relevancia,4.000,1.000,0.758,1.000,Sí
9,A. Variables predictoras,A7,Actividad física recreativa,Claridad,4.000,1.000,0.758,1.000,Sí


## Resumen por ítem y por sección

In [4]:
if HAY_DATOS:
    por_item = (resultado.groupby(['seccion', 'item_id', 'nombre'], sort=False)
                .agg(V_minimo=('V', 'min'), V_promedio=('V', 'mean'))
                .reset_index())
    por_item['valido (todos los criterios >= 0.80)'] = np.where(por_item['V_minimo'] >= V_MINIMO, 'Sí', 'No')
    por_item.to_csv(SALIDA / 'v_aiken_por_item.csv', index=False)
    display(por_item)
    por_seccion = resultado.groupby('seccion')['V'].agg(['mean', 'min']).rename(columns={'mean': 'V_promedio', 'min': 'V_minimo'})
    por_seccion.to_csv(SALIDA / 'v_aiken_por_seccion.csv')
    print(f"V de Aiken global: {resultado['V'].mean():.3f}")
    display(por_seccion)

,seccion,item_id,nombre,V_minimo,V_promedio,valido (todos los criterios >= 0.80)
0,A. Variables predictoras,A1,Edad,1.000,1.000,Sí
1,A. Variables predictoras,A2,Sexo,0.917,0.972,Sí
2,A. Variables predictoras,A5,Hipertensión diagnosticada,1.000,1.000,Sí
3,A. Variables predictoras,A7,Actividad física recreativa,1.000,1.000,Sí
4,A. Variables predictoras,A8,Tabaquismo actual,1.000,1.000,Sí
5,B. Preguntas del formulario,B2,Embarazo (solo mujeres),0.917,0.972,Sí
6,B. Preguntas del formulario,B3,Edad,0.833,0.944,Sí
7,B. Preguntas del formulario,B8,Actividad física,0.917,0.944,Sí
8,B. Preguntas del formulario,B9,Tabaco,0.833,0.944,Sí
9,C. Mensajes del resultado,C3,Nota sobre las contribuciones,0.833,0.944,Sí


V de Aiken global: 0.968


,V_promedio,V_minimo
seccion,,
A. Variables predictoras,0.994,0.917
B. Preguntas del formulario,0.951,0.833
C. Mensajes del resultado,0.958,0.833
D. Catálogo de recomendaciones,0.938,0.833


## Ítems que requieren corrección

In [5]:
if HAY_DATOS:
    corregir = resultado[resultado['V'] < V_MINIMO]
    if corregir.empty:
        print('Todos los ítems alcanzan V >= 0.80 en todos los criterios.')
    else:
        print(f'{corregir.item_id.nunique()} ítems con algún criterio por debajo de 0.80. Revise las observaciones de los expertos en las fichas.')
        display(corregir)

Todos los ítems alcanzan V >= 0.80 en todos los criterios.


## Observaciones de los expertos

Observaciones por ítem, observaciones generales y opinión de aplicabilidad, transcritas de las fichas. Sirven para decidir las correcciones junto con la V de Aiken (los ítems con V < 0.80 se corrigen obligatoriamente).

In [6]:
if HAY_DATOS:
    if observaciones.empty:
        print('No se transcribieron observaciones por ítem.')
    else:
        obs = observaciones.merge(items[['item_id', 'nombre']], on='item_id')[['item_id', 'nombre', 'experto', 'observacion']]
        obs = obs.sort_values(['item_id', 'experto'])
        obs.to_csv(SALIDA / 'observaciones.csv', index=False)
        display(obs)
    generales.to_csv(SALIDA / 'observaciones_generales.csv', index=False)
    display(generales)

No se transcribieron observaciones por ítem.


,experto,observaciones_generales,aplicabilidad
0,E1,N/A,Aplicable
1,E2,N/A,Aplicable
2,E3,N/A,Aplicable
3,E4,N/A,Aplicable
